# THỬ NGHIỆM ĐẶC TRƯNG LBP VÀ GLCM TRÊN ẢNH CCCD MẪU
**Thành viên thực hiện**: TV4 - Trần Đức Mạnh (Đặc trưng kết cấu)  
**Mục tiêu (Task T06)**: Thử nghiệm trích xuất Local Binary Patterns (LBP) và Gray-Level Co-occurrence Matrix (GLCM) bằng `skimage` trên ảnh mẫu thật (Real) và ảnh mẫu can thiệp (Tampered).


In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops

# Thêm đường dẫn src
sys.path.append('..')
from src.preprocessing.align import preprocess_card
from src.features.lbp_extractor import LBPExtractor, extract_lbp
from src.features.glcm_extractor import GLCMExtractor, extract_glcm


## 1. Nạp ảnh mẫu thật (Real) và ảnh giả (Tampered)

In [ ]:
real_img_path = '../data/raw/real_0000.png'
fake_img_path = '../data/tampered/fake_splicing_0000.png'

real_img = cv2.imread(real_img_path)
fake_img = cv2.imread(fake_img_path)

_, real_gray = preprocess_card(real_img)
_, fake_gray = preprocess_card(fake_img)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(real_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Ảnh CCCD Thật (Real)")
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("Ảnh CCCD Giả mạo - Cắt ghép mặt (Splicing Fake)")
axes[1].axis('off')
plt.show()


## 2. Thử nghiệm trích xuất đặc trưng LBP (Local Binary Patterns)
Sử dụng Uniform LBP với bán kính $R=1$, số điểm $P=8$.
Quan sát ma trận LBP và Histogram phân bố để nhận diện sự khác biệt kết cấu vi mô:


In [ ]:
lbp_calc = LBPExtractor(n_points=8, radius=1, method='uniform')
real_lbp_map = lbp_calc.compute_lbp_map(real_gray)
fake_lbp_map = lbp_calc.compute_lbp_map(fake_gray)

real_lbp_hist = extract_lbp(real_gray)
fake_lbp_hist = extract_lbp(fake_gray)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].imshow(real_lbp_map, cmap='gray')
axes[0, 0].set_title("Ma trận LBP - Ảnh Thật")
axes[0, 0].axis('off')

axes[0, 1].imshow(fake_lbp_map, cmap='gray')
axes[0, 1].set_title("Ma trận LBP - Ảnh Giả mạo (vết cắt viền rõ ở ảnh chân dung)")
axes[0, 1].axis('off')

bins = np.arange(10)
axes[1, 0].bar(bins, real_lbp_hist[:10], color='teal', alpha=0.7)
axes[1, 0].set_title("Histogram LBP - Ảnh Thật")
axes[1, 0].set_xlabel("LBP Uniform Codes")

axes[1, 1].bar(bins, fake_lbp_hist[:10], color='crimson', alpha=0.7)
axes[1, 1].set_title("Histogram LBP - Ảnh Giả mạo")
axes[1, 1].set_xlabel("LBP Uniform Codes")

plt.tight_layout()
plt.show()


## 3. Thử nghiệm trích xuất đặc trưng GLCM (Gray-Level Co-occurrence Matrix)
Tính toán các chỉ số thống kê Haralick ở khoảng cách $d \in [1, 2, 3]$ và 4 hướng góc:
- Contrast (Độ tương phản)
- Homogeneity (Độ đồng nhất)
- Energy (Năng lượng)
- Correlation (Tương quan)


In [ ]:
real_glcm_feats = extract_glcm(real_gray)
fake_glcm_feats = extract_glcm(fake_gray)

props = ['Contrast (Mean)', 'Contrast (Std)', 'Dissimilarity (Mean)', 'Dissimilarity (Std)',
         'Homogeneity (Mean)', 'Homogeneity (Std)', 'Energy (Mean)', 'Energy (Std)',
         'Correlation (Mean)', 'Correlation (Std)', 'ASM (Mean)', 'ASM (Std)']

import pandas as pd
df_comp = pd.DataFrame({
    'Thuộc tính Haralick': props,
    'Ảnh Thật (Real)': [f"{v:.4f}" for v in real_glcm_feats],
    'Ảnh Giả (Tampered)': [f"{v:.4f}" for v in fake_glcm_feats]
})
df_comp


## 4. Kết luận sơ bộ cho Báo cáo (TV4)
- **LBP**: Rất nhạy với các vết cắt viền xung quanh ảnh chân dung và sự mất cân xứng vi mô do làm mờ (smoothing).
- **GLCM**: Hoa văn bảo an Guilloche trên thẻ thật có tính tuần hoàn cao dẫn đến độ đồng nhất (Homogeneity) và tương quan (Correlation) ổn định, trong khi vùng bị cắt ghép làm tụt giảm độ đồng nhất.
- Kết hợp cả 2 nguồn đặc trưng giúp phát hiện bất thường kết cấu rất hiệu quả trước khi đưa vào bộ phân loại.
